In [46]:
# Mini Recommendation System using K-Means Clustering (Unsupervised Learning)

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

In [47]:
# Step 1: Load dataset (MovieLens small dataset - simulating Netflix data)
url_ratings = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv"
ratings = pd.read_csv(url_ratings)

In [48]:
# Use a smaller sample for quick demo
ratings = ratings.sample(20000, random_state=42)

print("Sample Ratings Data:")
print(ratings.head())

Sample Ratings Data:
         user_id  book_id  rating
3623535    42562     2757       3
3985638    43232      134       4
2983642    37244     1463       5
5812251    53366       71       2
2208852    29634     3339       4


In [49]:
# Step 2: Create User-Movie Matrix
user_movie_matrix = ratings.pivot_table(
    index="user_id",
    columns="book_id",   # treating book_id ~ movie_id
    values="rating"
).fillna(0)

In [50]:
print("\nUser-Movie Matrix Shape:", user_movie_matrix.shape)



User-Movie Matrix Shape: (16538, 6078)


In [51]:
# Step 3: Apply K-Means clustering on users
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
user_movie_matrix['Cluster'] = kmeans.fit_predict(user_movie_matrix)

print("\nCluster distribution:")
print(user_movie_matrix['Cluster'].value_counts())


Cluster distribution:
Cluster
2    16402
4       87
3       24
1       17
0        8
Name: count, dtype: int64


In [52]:
# Step 4: Pick a target user (simulate a Netflix subscriber)
target_user = 15
user_cluster = user_movie_matrix.iloc[target_user] ['Cluster']
print(f"\nTarget User {target_user} belongs to Cluster {user_cluster}")


Target User 15 belongs to Cluster 2.0


In [53]:
user_cluster = user_movie_matrix[user_movie_matrix['Cluster'] == 2]
print(user_cluster)

book_id    1    2    3    4    5    6    7    8    9   10  ...  9972  9974  \
user_id                                                    ...               
4        0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
6        0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
7        0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
12       0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
13       0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
...      ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   ...   ...   
53401    0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
53406    0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
53416    0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
53421    0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
53422    0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  

In [55]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: Load dataset (MovieLens small dataset - simulating Netflix data)
url_ratings = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv"
ratings = pd.read_csv(url_ratings)
# Use a smaller sample for quick demo
ratings = ratings.sample(20000, random_state=42)

print("Sample Ratings Data:")
print(ratings.head())

# Step 2: Create User-Movie Matrix
user_movie_matrix = ratings.pivot_table(
    index="user_id",
    columns="book_id",   # treating book_id ~ movie_id
    values="rating"
).fillna(0)
print("\nUser-Movie Matrix Shape:", user_movie_matrix.shape)

# Step 3: Apply K-Means clustering on users
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
user_movie_matrix['Cluster'] = kmeans.fit_predict(user_movie_matrix)

print("\nCluster distribution:")
print(user_movie_matrix['Cluster'].value_counts())

# Step 4: Pick a target user (simulate a Netflix subscriber)
target_user = 15
user_cluster = user_movie_matrix.iloc[target_user]['Cluster']
print(f"\nTarget User {target_user} belongs to Cluster {user_cluster}")

# Filter out users in the same cluster
similar_users = user_movie_matrix.loc[
    user_movie_matrix["Cluster"] == user_cluster
].drop(columns="Cluster")

# Step 5: Find similar users from the same cluster
target_user_ratings = user_movie_matrix.iloc[target_user].drop('Cluster')
similarities = cosine_similarity([target_user_ratings], similar_users)[0]
similarities_df = pd.DataFrame({
    "UserID": similar_users.index,
    "Similarity": similarities
}).sort_values(by="Similarity", ascending=False)

print("\nTop 5 similar users:")
print(similarities_df.head())

# Step 6: Recommend movies
# Movies rated highly by similar users but not yet rated by target user
target_user_ratings = ratings[ratings['user_id'] == target_user]
watched_movies = set(target_user_ratings['book_id'])

# Find top-rated movies in the cluster
cluster_user_ids = similar_users.index
cluster_ratings = ratings[ratings['user_id'].isin(cluster_user_ids)]

movie_scores = cluster_ratings.groupby('book_id')['rating'].mean().sort_values(ascending=False)

# Recommend top-rated movies that the target user hasn't watched yet
recommendations = [movie for movie in movie_scores.index if movie not in watched_movies][:5]

print(f"\n Recommended Movies for User {target_user}: {recommendations}")


Sample Ratings Data:
         user_id  book_id  rating
3623535    42562     2757       3
3985638    43232      134       4
2983642    37244     1463       5
5812251    53366       71       2
2208852    29634     3339       4

User-Movie Matrix Shape: (16538, 6078)

Cluster distribution:
Cluster
2    16402
4       87
3       24
1       17
0        8
Name: count, dtype: int64

Target User 15 belongs to Cluster 2.0

Top 5 similar users:
       UserID  Similarity
15         42         1.0
16377   53345         0.0
16369   53320         0.0
25         68         0.0
16401   53422         0.0

 Recommended Movies for User 15: [4690, 9822, 9878, 9875, 9874]


In [56]:
similarities = cosine_similarity([similar_users.iloc[target_user]], similar_users)[0]
similarities_df = pd.DataFrame({
    "UserID": similar_users.index,
    "Similarity": similarities
}).sort_values(by="Similarity", ascending=False)

print("\nTop 5 similar users:")
print(similarities_df.head())



Top 5 similar users:
       UserID  Similarity
15         42         1.0
16377   53345         0.0
16369   53320         0.0
25         68         0.0
16401   53422         0.0


In [57]:
# Step 6: Recommend movies
# Movies rated highly by similar users but not yet rated by target user
target_user_ratings = ratings[ratings['user_id'] == target_user]
watched_movies = set(target_user_ratings['book_id'])

In [59]:
# Find top-rated movies in the cluster
cluster_user_ids = similar_users.index
cluster_ratings = ratings[ratings['user_id'].isin(cluster_user_ids)]

movie_scores = cluster_ratings.groupby('book_id')['rating'].mean().sort_values(ascending=False)


In [60]:
recommendations = [movie for movie in movie_scores.index if movie not in watched_movies][:5]

print(f"\n Recommended Movies for User {target_user}: {recommendations}")


 Recommended Movies for User 15: [4690, 9822, 9878, 9875, 9874]
